In [ ]:
# %% Deep learning - Section 19.181
#    Code challenge 29: AEs on occluded Gaussians
#
#    1) Start from code from video 19.180 (gaussian synthetic dataset)
#    2) Try to make the model so that the AE removes the occlsion bar
#    3) Tried a) training with mixed data and b) using the non-occluded as the
#       target variable in the loss function

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.


In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [98]:
# %% Generate data

# 2D Gaussian params
n_gaussians = 1000
img_size    = 91
x           = np.linspace(-4,4,img_size)
X,Y         = np.meshgrid(x,x)

# Vary FWHM smoothly
widths = np.linspace(2,20,n_gaussians)

# Preallocate tensors for images (2*N,channels,size,size)
images   = torch.zeros(2*n_gaussians,1,img_size,img_size)

# Generate images
for i in range(n_gaussians):

    # Gaussian with random center offset
    c = 1.5*np.random.randn(2)
    G  = np.exp( -( (X-c[0])**2 + (Y-c[1])**2) / widths[i] )

    # Layer some noise
    G  = G + np.random.randn(img_size,img_size)/5

    # Add to tensor (occluded)
    images[i,:,:,:] = torch.Tensor(G).view(1,img_size,img_size)

    # Add random bar randomly
    i1 = np.random.choice(np.arange(2,28))
    i2 = np.random.choice(np.arange(2,6))
    if np.random.randn()>0:
        G[i1:i1+i2,] = 1
    else:
        G[:,i1:i1+i2] = 1

    # Add to tensor (occluded)
    images[i+n_gaussians,:,:,:] = torch.Tensor(G).view(1,img_size,img_size)


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5))/2
fig,axs = plt.subplots(2,7,figsize=(phi*7,7))

for i in range(7):

    pic = np.random.randint(n_gaussians)
    tmp = pic + n_gaussians
    G1  = np.squeeze( images[pic,:,:] )
    G2  = np.squeeze( images[tmp,:,:] )

    axs[0,i].imshow(G1,vmin=-1,vmax=1,cmap='jet')
    axs[1,i].imshow(G2,vmin=-1,vmax=1,cmap='jet')

    axs[0,i].set_xticks([])
    axs[0,i].set_yticks([])
    axs[1,i].set_xticks([])
    axs[1,i].set_yticks([])

plt.tight_layout()
plt.suptitle('Occluded and non-occluded Gaussians')

plt.savefig('figure60_code_challenge_29.png')
plt.show()
files.download('figure60_code_challenge_29.png')


In [34]:
# %% Function to generate the model

# Basically for the decoder we do "transpose" convolution
def gen_model():

    class Gauss_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            # Encoding layer
            self.encoder = nn.Sequential( nn.Conv2d(1,6,3,padding=1),
                                          nn.ReLU(),
                                          nn.MaxPool2d(2,2),
                                          nn.Conv2d(6,4,3,padding=1),
                                          nn.ReLU(),
                                          nn.MaxPool2d(2,2) )

            # Decoding layer
            self.decoder = nn.Sequential( nn.ConvTranspose2d(4,6,3,2),
                                          nn.ReLU(),
                                          nn.ConvTranspose2d(6,1,3,2) )

        def forward(self,x):
            return self.decoder(self.encoder(x))

    # Create model instance
    CNN = Gauss_CNN()

    # Loss function
    loss_fun = nn.MSELoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [ ]:
# %% Test the model on one batch

CNN,loss_fun,optimizer = gen_model()
yHat = CNN(images[:10,:,:,:])

print()
print(yHat.shape)

# Plotting
fig,ax = plt.subplots(1,2,figsize=(8,3))
ax[0].imshow(torch.squeeze(images[0,0,:,:]).detach(),cmap='jet')
ax[0].set_title('Model input')
ax[1].imshow(torch.squeeze(yHat[0,0,:,:]).detach(),cmap='jet')
ax[1].set_title('Model output')

plt.suptitle('Model test')
plt.tight_layout()

plt.savefig('figure61_code_challenge_29.png')
plt.show()
files.download('figure61_code_challenge_29.png')


In [ ]:
# %% Check all the parameters in the model

summary(CNN,(1,img_size,img_size))


In [72]:
# %% Function to train the model

# Train on both occluded and non-occluded data (~10% occluded)

def train_model():

    # Parameters, model instance, inizialise vars
    num_epochs = 500
    CNN,loss_fun,optimizer = gen_model()

    losses = []

    # Loop over epochs
    for epoch_i in range(num_epochs):

        # pick a set of images at random (double gaussians)
        pics_batch = np.random.choice(n_gaussians+int(n_gaussians*0.1),size=32,replace=False)
        X = images[pics_batch,:,:,:]

        # Forward propagation and loss
        yHat = CNN(X)
        loss = loss_fun(yHat,X)

        losses.append(loss.item())

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return losses,CNN


In [99]:
# %% Function to train the model

# Alternative: use non-occluded as target in loss (increase epochs)

def train_model():

    # Parameters, model instance, inizialise vars
    num_epochs = 1000
    CNN,loss_fun,optimizer = gen_model()

    losses = []

    # Loop over epochs
    for epoch_i in range(num_epochs):

        # pick a set of images at random (double gaussians)
        pics_batch = np.random.choice(n_gaussians,size=32,replace=False)
        X_clean = images[pics_batch,:,:,:]
        X_occl  = images[pics_batch+n_gaussians,:,:,:]

        # Forward propagation and loss
        yHat = CNN(X_occl)
        loss = loss_fun(yHat,X_clean)

        losses.append(loss.item())

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return losses,CNN


In [100]:
# %% Run the model

losses,CNN = train_model()


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(phi*5,5))

plt.plot(losses,'-',label='Train')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.title('Model loss (final loss=%.3f)'%losses[-1])

plt.savefig('figure62_code_challenge_29.png')
plt.show()
files.download('figure62_code_challenge_29.png')


In [ ]:
# %% Plotting

# Select only from the last 1000 pics occluded images only
pics_batch = np.random.choice(np.arange(n_gaussians,2*n_gaussians),size=32,replace=False)
X = images[pics_batch,:,:,:]
yHat = CNN(X)

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(2,10,figsize=(1.5*phi*6,6))

for i in range(10):

    G = torch.squeeze( X[i,0,:,:] ).detach()
    O = torch.squeeze( yHat[i,0,:,:] ).detach()

    axs[0,i].imshow(G,vmin=-1,vmax=1,cmap='jet')
    axs[0,i].axis('off')
    axs[0,i].set_title('Model input')

    axs[1,i].imshow(O,vmin=-1,vmax=1,cmap='jet')
    axs[1,i].axis('off')
    axs[1,i].set_title('Model output')

plt.tight_layout()

plt.savefig('figure63_code_challenge_29.png')
plt.show()
files.download('figure63_code_challenge_29.png')


In [ ]:
# %% Exercise 1
#    The network does OK but there are still residual occlusion artifacts. Perhaps there weren't enough training examples?
#    If you would increase nGauss from 1000 to 10000, would that mean that the model trains on 10x as many examples?
#    (Hint: the answer is No, but you need to figure out why!) How can you adapt the model so that it trains on more
#    unique sample images?

# No it means we just pick randomly from a larger pool, if we wanted the model
# to train on unique images we should use a DataLoader and pass unique
# batches of images


In [93]:
# %% Exercise 2
#    The bars appear in a random location for each image. Would the network still learn to remove the occlusions if the
#    bars appeared in the exact same location with the same thickness? Change the stimulus generation code to implement
#    this. You can still keep the randomization to horizontal or vertical, but remove the random selection of thickness
#    and location.

# So, apparently it helps a bit; possibly the fixed occlusion is removed more
# easily because it's a more stable feature

# Generate data
n_gaussians = 1000
img_size    = 91
x           = np.linspace(-4,4,img_size)
X,Y         = np.meshgrid(x,x)

# Vary FWHM smoothly
widths = np.linspace(2,20,n_gaussians)

# Preallocate tensors for images (2*N,channels,size,size)
images   = torch.zeros(2*n_gaussians,1,img_size,img_size)

# Constant bar location and width
i1 = np.random.choice(np.arange(2,28))
i2 = np.random.choice(np.arange(2,6))

# Generate images
for i in range(n_gaussians):

    # Gaussian with random center offset
    c = 1.5*np.random.randn(2)
    G  = np.exp( -( (X-c[0])**2 + (Y-c[1])**2) / widths[i] )

    # Layer some noise
    G  = G + np.random.randn(img_size,img_size)/5

    # Add to tensor (occluded)
    images[i,:,:,:] = torch.Tensor(G).view(1,img_size,img_size)

    # Add random bar randomly
    if np.random.randn()>0:
        G[i1:i1+i2,] = 1
    else:
        G[:,i1:i1+i2] = 1

    # Add to tensor (occluded)
    images[i+n_gaussians,:,:,:] = torch.Tensor(G).view(1,img_size,img_size)

